# Qwen SAE Neuron Viewer
Runs `simple_server.py` with the updated neuron DB and exposes it via localtunnel.

**Runtime:** GPU (T4 or better)

In [ ]:
# 1. Dependencies
!pip install -q huggingface_hub transformers torch nnsight pyngrok
!npm install -g localtunnel -q

In [ ]:
# 2. HF login (set HF_TOKEN in Colab secrets, or paste token when prompted)
from huggingface_hub import login
login()

In [ ]:
# 3. Clone the repo (has updated neuron DB + latest code)
!git clone https://github.com/p3rciv3l/winnie-the-pooh-qwen.git /content/repo

In [ ]:
# 4. Download SAE checkpoints + activation data from HF
#    We skip neuron_db — the repo already has the updated version.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="OysterAI/Qwen2.5-3B-Instruct-SAEs",
    local_dir="/content/sae_hf",
    ignore_patterns=["*.py", "*.txt", "*.md", "data/neuron_db/*"],
)

In [ ]:
# 5. Move SAE data into the repo's data/ directory
import shutil, os

for subdir in ["sae_checkpoints", "activation"]:
    src = f"/content/sae_hf/data/{subdir}"
    dst = f"/content/repo/data/{subdir}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Copied {subdir}")
    else:
        print(f"WARNING: {src} not found — check HF repo structure")

In [ ]:
# 6. Download Qwen2.5-3B-Instruct
snapshot_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    local_dir="/content/Qwen2.5-3B-Instruct",
)

In [ ]:
# 7. Launch the server
import os, subprocess

os.environ["SOURCE_MODEL"] = "/content/Qwen2.5-3B-Instruct"

server = subprocess.Popen(
    ["python", "simple_server.py"],
    cwd="/content/repo",
)
print("Server started (PID", server.pid, ")")

In [ ]:
# 8. Expose via localtunnel
#    The printed URL is your public endpoint. 
#    If it asks for a password, run the next cell to get the tunnel password.
import time
time.sleep(15)  # wait for server to finish loading model

!lt --port 9999 &

In [ ]:
# Tunnel password (your Colab IP — paste this if localtunnel prompts)
!curl -s https://ipv4.icanhazip.com